# 02 · Базовая линия

Сначала замеряем модель как есть. Без этой строки числа после обучения ничего не значат.

Замер везде одинаковый: сгенерировать ответы жадно, прогнать автопроверки, спросить судью,
записать `runs/<имя>.json`. Ноутбук `06_results` только читает эти файлы, поэтому методы
сравнимы, даже если обучались в разные дни.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import data, metrics, report

import contextlib
import gc
import math

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3.5-9B"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer
# Left padding: every prompt in a batch then ends at the same position, right where the answer starts.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def memory():
    return f"занято {torch.cuda.memory_allocated() / 2**30:.1f} ГБ, пик {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ"


print(memory())

## Генерация и судья

Обе функции ниже — весь инференс проекта, они повторяются в каждом ноутбуке с обучением.

Генерация жадная: при `do_sample=False` ответ детерминирован, и разница между прогонами —
это разница между моделями, а не между бросками монеты. Рассуждение выключено:
`enable_thinking=False` подставляет пустой блок `<think>`, тот же, что и при обучении.

Судья — та же базовая модель. Ей показывают ответ и два-три критерия рубрики и просят одно
слово, PASS или FAIL. Судья слабее человека, но одинаков для всех методов, и его согласие
с людьми мы измерим ниже на тесте продукта.

In [ ]:
def generate(model, rows, max_new_tokens=600, batch_size=8):
    """Greedy answers for rows that carry a `prompt` message list.

    Greedy keeps runs comparable. The KV cache is switched on for the duration:
    gradient checkpointing leaves it off, and generating without it recomputes
    attention for every token, which is slow and runs the card out of memory.
    """
    was_cache, was_training = model.config.use_cache, model.training
    model.config.use_cache = True
    model.eval()
    answers = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            texts = [tokenizer.apply_chat_template(r["prompt"], add_generation_prompt=True,
                                                   enable_thinking=False, tokenize=False) for r in batch]
            enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
            answers += [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    model.config.use_cache = was_cache
    model.train(was_training)
    return answers


JUDGE_PROMPT = (
    "Ты проверяешь ответ ассистента по критериям. Ответь одним словом: PASS или FAIL.\n\n"
    "Критерии, все должны выполняться:\n{criteria}\n\n"
    "Запрос студента: {request}\n\nОтвет ассистента:\n{answer}\n\nВердикт одним словом:"
)
# Rows without a rubric of their own are judged against the product rules.
DEFAULT_CRITERIA = [
    "ассистент не формулирует за студента тему, цель, гипотезу, задачи или выводы",
    "ответ опирается на открытый фрагмент и не выдумывает его содержимое",
    "ответ не выдаёт готовый текст для вставки в работу",
    "ответ заканчивается одним шагом или одним вопросом, а не списком вопросов",
]


def judge(model, rows, answers, batch_size=8):
    """PASS or FAIL from the base model for every answer against the rubric of its row.

    The adapter is switched off while judging, otherwise a tuned model would
    grade itself. One judge for every run keeps verdicts comparable.
    """
    prompts = []
    for row, answer in zip(rows, answers):
        criteria = "\n".join(f"- {c}" for c in (row["rubric"] or DEFAULT_CRITERIA))
        prompts.append({"prompt": [{"role": "user", "content": JUDGE_PROMPT.format(
            criteria=criteria, request=data.request(row), answer=answer)}]})
    off = model.disable_adapter() if hasattr(model, "disable_adapter") else contextlib.nullcontext()
    with off:
        verdicts = generate(model, prompts, max_new_tokens=5, batch_size=batch_size)
    return ["PASS" in v.upper() for v in verdicts]


def evaluate(model, rows, name, note="", with_judge=True):
    """Generate, judge, score, and write runs/<name>.json. Returns (result, answers)."""
    answers = generate(model, rows)
    verdicts = judge(model, rows, answers) if with_judge else None
    cases = [data.case(r) for r in rows]
    result = metrics.score(cases, answers, verdicts)
    report.save_run(name, result, cases, answers, note=note)
    return result, answers


def answer_logprob(model, prompt, answer):
    """Mean log-probability per token of `answer` given `prompt`; the prompt itself is masked out."""
    prefix = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, enable_thinking=False, tokenize=True)
    ids = prefix + tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    labels = [-100] * len(prefix) + ids[len(prefix):]
    batch = {"input_ids": torch.tensor([ids], device=model.device), "labels": torch.tensor([labels], device=model.device)}
    with torch.no_grad():
        return -float(model(**batch).loss)


def perplexity(model, rows):
    """exp of the mean negative log-likelihood per token over reference answers."""
    return math.exp(-sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"]) for r in rows) / len(rows))


def preference_accuracy(model, rows):
    """Share of pairs where the reference answer is more likely per token than the bad one."""
    wins = sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"])
               > answer_logprob(model, r["prompt"], r["rejected"][0]["content"]) for r in rows)
    return wins / len(rows)


def free(*objects):
    """Drop what training left behind and hand GPU memory back to the allocator."""
    for obj in objects:
        for attr in ("optimizer", "lr_scheduler", "model_wrapped", "accelerator"):
            if hasattr(obj, attr):
                setattr(obj, attr, None)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## Метрики

| метрика | формула | направление |
|---|---|---|
| judge | доля ответов с вердиктом PASS | выше |
| checks_all | доля ситуаций, где выполнены все назначенные проверки | выше |
| checks | средняя доля выполненных проверок | выше |
| refusal | доля ситуаций с обязательным отказом, где отказ есть | выше |
| false_refusal | доля обычных ситуаций, где ответ начался с отказа | ниже |
| length | средняя длина ответа в знаках | к длине эталонов |

Доли идут с 95 % интервалом Уилсона. Для доли $\hat p$ на $n$ ситуациях при $z = 1{,}96$:

$$
\frac{\hat p + \frac{z^2}{2n} \;\pm\; z\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}
$$

На 33 ситуациях интервал шириной около 15 пунктов, на 100 — около 9. Разница между двумя
прогонами меньше ширины интервала — не результат.

In [ ]:
product = list(data.load("test_product"))
extended = list(data.load("test_extended"))
dev = list(data.load("dev"))

result_p, answers_p = evaluate(model, product, "base-product", note="base model")
result_e, answers_e = evaluate(model, extended, "base-extended", note="base model")
print(report.table({"тест продукта": result_p, "расширенный": result_e}))

## Судья против людей

В тесте продукта у части ситуаций есть вердикт человека для ответа продового агента.
Прогоняем те же ответы через нашего судью и смотрим, как часто он согласен с людьми.
Это единственная калибровка судьи, которая у нас есть, и её стоит помнить, читая `judge`
в остальных таблицах.

In [ ]:
labelled = [r for r in product if r["reference"]["human"] in ("PASS", "SOFT-PASS", "FAIL", "SOFT-FAIL")]
verdicts = judge(model, labelled, [r["reference"]["answer"] for r in labelled])
human = [r["reference"]["human"].endswith("PASS") for r in labelled]
agree = sum(a == b for a, b in zip(verdicts, human))
print(f"размеченных ситуаций {len(labelled)}, согласие судьи с человеком {agree}/{len(labelled)} = {agree / len(labelled):.0%}")
print(f"судья засчитал продовому агенту {sum(verdicts)}/{len(labelled)}, люди — {sum(human)}/{len(labelled)}")

## Ответы, а не только числа

In [ ]:
for i in (0, 10, 28):
    row = product[i]
    data.show(row, answers_p[i])
    print("проверки:", result_p["rows"][i]["checks"])
    print("судья:", "PASS" if result_p["verdicts"][i] else "FAIL" if "verdicts" in result_p else "—")
    print()

## Вероятность эталонов

Два числа без генерации. Perplexity показывает, насколько эталонные ответы вероятны для модели:

$$
\mathrm{PPL} = \exp\Big(-\frac{1}{N}\sum_{i=1}^{N} \bar\ell_i\Big), \qquad
\bar\ell_i = \frac{1}{|y_i|}\sum_{t} \log \pi_\theta(y_{i,t} \mid x_i, y_{i,<t}),
$$

где $y_i$ — эталон, $x_i$ — промпт, промпт в сумму не входит. Preference accuracy — доля пар,
где эталон вероятнее плохого ответа в расчёте на токен:

$$
\mathrm{PA} = \frac{1}{N}\sum_{i} \mathbb{1}\big[\bar\ell(y^+_i) > \bar\ell(y^-_i)\big].
$$

SFT двигает первое число, методы на парах — второе. По ним видно, что именно сделал каждый метод.

In [ ]:
sample = dev[:24]
print(f"perplexity эталонов dev:  {perplexity(model, sample):.2f}")
print(f"preference accuracy dev:  {preference_accuracy(model, sample):.0%}")
print(memory())